In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
print("All imports loaded successfully.")

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    def __init__(self, d_model, max_seq_length=512):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))   # [1, max_seq, d_model]

    def forward(self, x):
        # x: [B, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k       = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        B, _, T, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(B, T, self.num_heads * d_k)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, -1e9)
        attn = torch.softmax(scores, dim=-1)
        return self.W_o(self.combine_heads(torch.matmul(attn, V)))


In [ ]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.3):
        super().__init__()
        self.fc1     = nn.Linear(d_model, d_ff)
        self.fc2     = nn.Linear(d_ff, d_model)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(self.relu(self.fc1(x))))

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = PointWiseFeedForward(d_model, d_ff, dropout)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn       = MultiHeadAttention(d_model, num_heads)
        self.ffn              = PointWiseFeedForward(d_model, d_ff, dropout)
        self.norm1            = nn.LayerNorm(d_model)
        self.norm2            = nn.LayerNorm(d_model)
        self.norm3            = nn.LayerNorm(d_model)
        self.dropout          = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None):
        # 1. Masked Self-Attention + Add & LayerNorm
        x = self.norm1(x + self.dropout(
            self.masked_self_attn(x, x, x, tgt_mask)
        ))
        # 2. Cross-Attention (Q=decoder, K=V=encoder) + Add & LayerNorm
        x = self.norm2(x + self.dropout(
            self.cross_attn(x, enc_output, enc_output, None)
        ))
        # 3. FFN + Add & LayerNorm
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x

In [ ]:
class TransformerNIDS(nn.Module):
    """
    Transformer-based NIDS for CIC-IDS 2018.
      - Each tabular row (num_features) is treated as a SEQUENCE of tokens.
      - Each scalar feature is embedded to d_model dimensions.
      - Input tensor: [B, num_features]  →  reshaped internally to [B, num_features, 1]
                                          →  projected to [B, num_features, d_model].
      - Dataset is balanced: standard CrossEntropyLoss used (no FocalLoss).
    """
    def __init__(self, input_dim, num_classes, d_model=32, num_heads=4,
                 num_encoder_layers=6, num_decoder_layers=6,
                 d_ff=1024, dropout=0.3, max_seq_length=512):
        super().__init__()
        self.encoder_embedding = nn.Linear(1, d_model)
        self.decoder_embedding = nn.Linear(1, d_model)
        self.pos_encoding      = PositionalEncoding(d_model, max_seq_length)
        self.emb_dropout       = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        # Output raw logits — CrossEntropyLoss handles softmax internally
        self.fc_out  = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def _causal_mask(self, seq_len, device):
        return torch.triu(
            torch.ones(seq_len, seq_len, device=device), diagonal=1
        ).bool()

    def forward(self, x):
        # x: [B, num_features]  (flat tabular row)
        x = x.unsqueeze(-1)           # [B, num_features, 1]

        # Encoder
        src = self.emb_dropout(self.pos_encoding(self.encoder_embedding(x)))
        enc_output = src
        for layer in self.encoder_layers:
            enc_output = layer(enc_output)

        # Decoder (with causal mask)
        tgt = self.emb_dropout(self.pos_encoding(self.decoder_embedding(x)))
        tgt_mask   = self._causal_mask(x.size(1), x.device)
        dec_output = tgt
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, tgt_mask)

        # Mean-pool over sequence dim → classify
        out = dec_output.mean(dim=1)   # [B, d_model]
        return self.fc_out(out)         # raw logits [B, num_classes]

print("TransformerNIDS defined (balanced dataset — CrossEntropyLoss, no FocalLoss).")


In [ ]:
# ── Load preprocessed CSV files ──────────────────────────────────────
train_df = pd.read_csv("processed/train.csv")
val_df   = pd.read_csv("processed/val.csv")
test_df  = pd.read_csv("processed/test.csv")

# ── Encode labels ─────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(np.concatenate([
    train_df["Label"].values,
    val_df["Label"].values,
    test_df["Label"].values
]))

# ── Build tensors ─────────────────────────────────────────────────────
X_train = torch.tensor(train_df.drop(columns=["Label"]).values, dtype=torch.float32)
y_train = torch.tensor(le.transform(train_df["Label"]), dtype=torch.long)

X_val   = torch.tensor(val_df.drop(columns=["Label"]).values, dtype=torch.float32)
y_val   = torch.tensor(le.transform(val_df["Label"]), dtype=torch.long)

# ── DataLoaders ───────────────────────────────────────────────────────
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=256, shuffle=False, num_workers=0)

# ── Device ───────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {device}")

# ── Model setup ───────────────────────────────────────────────────────
# Dataset is balanced → standard CrossEntropyLoss (no class weights, no FocalLoss)
input_dim   = X_train.shape[-1]    # number of feature columns
num_classes = len(le.classes_)
EPOCHS      = 25

model     = TransformerNIDS(input_dim=input_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()  # balanced data → no weighting needed

# Adam with OneCycleLR scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1          # 10% warmup
)

print(f"Input dim   : {input_dim}  (each feature = one sequence position)")
print(f"Num classes : {num_classes}")
print(f"Classes     : {list(le.classes_)}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"Optimizer   : Adam (lr=1e-3, weight_decay=1e-4)")
print(f"Scheduler   : OneCycleLR (max_lr=1e-3, 10% warmup, {EPOCHS} epochs)")
print(f"Loss        : CrossEntropyLoss (balanced dataset — no FocalLoss, no class weights)")


In [ ]:
model = model.to(device)

best_val_acc = 0.0
for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()    # OneCycleLR steps every BATCH
        train_loss    += loss.item()
        train_correct += (logits.argmax(1) == y_batch).sum().item()
        train_total   += len(y_batch)

    # ── Validate ──────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            val_loss    += loss.item()
            val_correct += (logits.argmax(1) == y_batch).sum().item()
            val_total   += len(y_batch)

    val_acc = val_correct / val_total * 100
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model_2018.pt")

    current_lr = scheduler.get_last_lr()[0]
    print(
        f"Epoch [{epoch+1:2d}/{EPOCHS}] "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss/len(train_loader):.4f}  Acc: {train_correct/train_total*100:.2f}% | "
        f"Val Loss: {val_loss/len(val_loader):.4f}  Acc: {val_acc:.2f}%"
    )

print(f"Best Val Accuracy: {best_val_acc:.2f}%")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load("best_model_2018.pt", map_location=device))

X_test_t    = torch.tensor(test_df.drop(columns=["Label"]).values, dtype=torch.float32)
y_test_t    = torch.tensor(le.transform(test_df["Label"]), dtype=torch.long)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device))
        preds  = logits.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    zero_division=0
))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.title("Confusion Matrix — TransformerNIDS on CIC-IDS 2018")
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig("confusion_matrix_2018.png", dpi=150)
plt.show()
print("Saved confusion_matrix_2018.png")
